# 02 — Baseline Models

**FairLens AI / NyayaLens — Person 3 (ML Layer)**

This notebook preprocesses the Adult dataset and trains a baseline
Logistic Regression model. We evaluate accuracy, precision, recall, and F1
**before** any fairness analysis or mitigation.

### What we do here
1. Load and preprocess the dataset (clean, encode, scale, split)
2. Train a Logistic Regression (max_iter=5000, random_state=42)
3. Evaluate performance metrics
4. Inspect the confusion matrix

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
print("Libraries loaded.")

## Step 1 — Load and preprocess

We repeat the same preprocessing from the hardcoded demo:
- Drop rows with `?` or NaN
- Encode target: `>50K` = 1, `<=50K` = 0
- Separate the sensitive column `sex` (keep it for later, don't feed to model)
- Drop `sex`, `race`, `fnlwgt` from features
- One-hot encode categoricals
- StandardScale all features
- 80/20 train/test split

In [ ]:
# --- Load ---
adult_bunch = fetch_openml("adult", version=2, as_frame=True)
raw_df = adult_bunch.frame
print(f"Raw shape: {raw_df.shape}")

# --- Clean ---
cleaned_df = raw_df.copy()
for col in cleaned_df.columns:
    if cleaned_df[col].dtype == object:
        cleaned_df = cleaned_df[cleaned_df[col] != "?"]
cleaned_df = cleaned_df.dropna()
print(f"After cleaning: {cleaned_df.shape[0]} rows")

# --- Encode target ---
income_labels = cleaned_df["class"].apply(
    lambda value: 1 if ">50K" in str(value) else 0
)
print(f"Target distribution: {income_labels.value_counts().to_dict()}")

# --- Separate sensitive column ---
gender_sensitive = cleaned_df["sex"].copy()

# --- Build features ---
feature_df = cleaned_df.drop(columns=["sex", "race", "fnlwgt", "class"])
feature_df = pd.get_dummies(feature_df, drop_first=True)
print(f"Feature shape after encoding: {feature_df.shape}")

# --- Split ---
features_train, features_test, labels_train, labels_test = train_test_split(
    feature_df, income_labels, test_size=0.2, random_state=42
)

# --- Scale ---
scaler = StandardScaler()
features_train = pd.DataFrame(
    scaler.fit_transform(features_train),
    columns=features_train.columns, index=features_train.index
)
features_test = pd.DataFrame(
    scaler.transform(features_test),
    columns=features_test.columns, index=features_test.index
)

# --- Match sensitive column to train/test ---
gender_train = gender_sensitive.loc[features_train.index]
gender_test = gender_sensitive.loc[features_test.index]

print(f"X_train: {features_train.shape}")
print(f"X_test:  {features_test.shape}")

## Step 2 — Train Logistic Regression

We use `max_iter=5000` because the Adult dataset with 91 one-hot features
needs more iterations to converge. `random_state=42` for reproducibility.

In [ ]:
baseline_model = LogisticRegression(max_iter=5000, random_state=42)
baseline_model.fit(features_train, labels_train)
print("Model trained successfully.")
print(f"Number of iterations: {baseline_model.n_iter_[0]}")

## Step 3 — Evaluate performance metrics

In [ ]:
baseline_predictions = baseline_model.predict(features_test)

accuracy  = accuracy_score(labels_test, baseline_predictions)
precision = precision_score(labels_test, baseline_predictions)
recall    = recall_score(labels_test, baseline_predictions)
f1        = f1_score(labels_test, baseline_predictions)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

## Step 4 — Confusion Matrix

The confusion matrix shows us True Positives, False Positives,
True Negatives, and False Negatives — helpful for understanding
where the model makes mistakes.

In [ ]:
cm = confusion_matrix(labels_test, baseline_predictions)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(cm, display_labels=["<=50K", ">50K"])
disp.plot(ax=ax, cmap="Blues")
ax.set_title("Baseline Model — Confusion Matrix")
plt.tight_layout()
plt.show()

print(f"\nTrue Negatives:  {cm[0][0]}")
print(f"False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]}")
print(f"True Positives:  {cm[1][1]}")

## Summary

- Baseline Logistic Regression achieves ~85% accuracy.
- Recall is lower than precision, meaning the model misses many true >50K earners.
- We haven't looked at fairness yet — that's Notebook 03.
- The variables `baseline_model`, `features_train/test`, `labels_train/test`,
  `gender_train/test` are all ready for the next notebook.

**Next:** Notebook 03 — Fairness Metrics